# RB AI 创作室（Google Colab 免费显卡版）

文字生图 → 人物/物品保持一致 → 图片生成视频，全部使用免费开源模型。

## 第一次使用前（只做一次）
1. **Hugging Face 账号**：到 https://huggingface.co 免费注册 → 右上角头像 → Settings → Access Tokens → 新建一个 **Read** 权限的 token，复制下来。
2. **同意视频模型的使用协议**：打开 https://huggingface.co/stabilityai/stable-video-diffusion-img2vid-xt ，登录后点同意（Agree）。不做这一步，生成视频会失败。
3. **把 token 存进 Colab**：点左边栏的 🔑（Secrets）→ 新增，名称填 `HF_TOKEN`，值贴上 token，并打开「Notebook access」。
4. 如果 rb-tools 仓库是**私有**的：再新增一个 Secret 叫 `GITHUB_TOKEN`，值是 GitHub 的 Personal Access Token（有读取仓库权限即可）。

## 每次使用
1. 菜单 **代码执行程序（Runtime）→ 更改运行时类型 → T4 GPU** → 保存。
2. 菜单 **代码执行程序 → 全部运行**。
3. 等最后一格出现 `https://xxxx.gradio.live` 链接，点开即可使用；把链接和密码发给同事也能一起用。
4. 用完关掉这个网页即可。免费版每天有使用时长限制，长时间不操作会自动断开。

In [ ]:
#@title ① 检查显卡
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "❌ 没有显卡：请到 代码执行程序 → 更改运行时类型 → 选 T4 GPU"

In [ ]:
#@title ② 设置
PASSWORD = "rb2026"  #@param {type:"string"}
#@markdown 登录用户名固定是 `rb`，密码就是上面这个。留空则不需要密码（不建议）。
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown 勾选后作品保存在 Google Drive 的 `AI_Studio` 文件夹，关掉 Colab 也不会丢。
BRANCH = "claude/sweet-turing-mb6b3k"  #@param {type:"string"}

In [ ]:
#@title ③ 安装（约 2-3 分钟）
import os, subprocess

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

hf_token = secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    print("⚠️ 没有找到 HF_TOKEN：图片可以生成，但生成视频会失败。请按上面说明设置。")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["AI_STUDIO_OUTPUT"] = "/content/drive/MyDrive/AI_Studio"
else:
    os.environ["AI_STUDIO_OUTPUT"] = "/content/outputs"

gh_token = secret("GITHUB_TOKEN")
auth = f"{gh_token}@" if gh_token else ""
repo = f"https://{auth}github.com/rbprosperitypurchasing-afk/rb-tools.git"
if os.path.isdir("/content/rb-tools"):
    subprocess.run(["git", "-C", "/content/rb-tools", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, repo, "/content/rb-tools"], check=True)

!pip install -q -r /content/rb-tools/ai_studio/requirements.txt
print("✅ 安装完成")

In [ ]:
#@title ④ 启动（出现 gradio.live 链接后点开使用）
#@markdown 第一次生成图片/视频时会下载模型（共约 15GB），需要几分钟，之后就快了。
%cd /content/rb-tools
!python -m ai_studio.app --share --password "{PASSWORD}"